### Code for collision risk analysis

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from skyfield.api import load, EarthSatellite

In [4]:
# 1. Configuration & Data Fetching
# You will need: pip install skyfield
CELESTRAK_URL = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=active&FORMAT=tle'

def get_orbit_regime(sat):
    """Classifies a Skyfield EarthSatellite object into LEO, MEO, or GEO."""
    # Mean motion is stored as radians per minute in the SGP4 model (no_kozai)
    # Conversion: (rad/min * 1440 min/day) / (2*pi rad/rev) = revs/day
    try:
        # Standard attribute for modern sgp4/skyfield
        mean_motion_rad_min = sat.model.no_kozai 
    except AttributeError:
        # Fallback for older versions
        mean_motion_rad_min = sat.model.no 

    revs_per_day = mean_motion_rad_min * 1440 / (2 * np.pi)
    
    if revs_per_day > 11.25:
        return "LEO"
    elif 1.05 < revs_per_day <= 11.25:
        return "MEO"
    elif 0.9 <= revs_per_day <= 1.05:
        return "GEO"
    else:
        return "HEO/Other"

def run_live_conjunction_analysis(target_norad_id, days=7, step_minutes=15):
    ts = load.timescale()
    now = datetime.utcnow()
    
    print(f"Fetching live TLEs from Celestrak...")
    try:
        satellites = load.tle_file(CELESTRAK_URL)
    except Exception as e:
        return f"Failed to download data: {e}"

    by_number = {sat.model.satnum: sat for sat in satellites}
    
    if target_norad_id not in by_number:
        return f"Error: Satellite {target_norad_id} not found in active list."
    
    target = by_number[target_norad_id]
    target_regime = get_orbit_regime(target)
    print(f"Target identified: {target.name}")
    print(f"Orbit Regime: {target_regime}")

    # Filter candidates by the same regime to reduce computational load
    candidates = [
        s for s in satellites 
        if s.model.satnum != target_norad_id and get_orbit_regime(s) == target_regime
    ]
    print(f"Scanning {len(candidates)} other {target_regime} objects for the next {days} days...")

    # Create time steps
    times = ts.utc(now.year, now.month, now.day, now.hour, 
                   range(0, int(days * 24 * 60), step_minutes))

    # Pre-calculate target trajectory
    target_pos = target.at(times).position.km
    
    results = []
    
    # Iterate through candidates and calculate Euclidean distances
    for sat in candidates:
        sat_pos = sat.at(times).position.km
        diff = target_pos - sat_pos
        distances = np.sqrt(np.sum(diff**2, axis=0))
        
        min_idx = np.argmin(distances)
        min_dist = distances[min_idx]
        
        results.append({
            'NAME': sat.name,
            'NORAD_ID': sat.model.satnum,
            'MIN_DIST_KM': round(min_dist, 2),
            'TIME_UTC': times[min_idx].utc_strftime('%Y-%m-%d %H:%M:%S')
        })

    # Sort by nearest distance
    top_10 = sorted(results, key=lambda x: x['MIN_DIST_KM'])[:10]
    return pd.DataFrame(top_10)

# Execute for ISS
df_results = run_live_conjunction_analysis(46355)
print("\nTop 10 Nearest Space Objects (Next 7 Days):")
print(df_results.to_string(index=False))

Fetching live TLEs from Celestrak...
Target identified: STARLINK-1661
Orbit Regime: LEO
Scanning 14063 other LEO objects for the next 7 days...

Top 10 Nearest Space Objects (Next 7 Days):
          NAME  NORAD_ID  MIN_DIST_KM            TIME_UTC
           HST     20580         8.70 2026-04-05 03:30:00
 STARLINK-2346     47887        12.34 2026-04-09 00:00:00
STARLINK-30554     58048        14.48 2026-04-05 08:00:00
STARLINK-31610     59473        16.47 2026-04-08 21:30:00
STARLINK-31546     59470        17.26 2026-04-08 02:45:00
STARLINK-30866     58237        19.44 2026-04-06 14:00:00
STARLINK-31039     58784        21.16 2026-04-06 08:45:00
STARLINK-31127     58765        24.11 2026-04-09 21:15:00
STARLINK-31073     58773        24.76 2026-04-07 20:45:00
 STARLINK-5961     56914        29.28 2026-04-05 11:45:00


In [6]:
# Execute for ISS
df_results = run_live_conjunction_analysis(46355)

import plotly.graph_objects as go

print("\nTop 10 Nearest Space Objects (Next 7 Days):")

# Handle errors or empty results
if isinstance(df_results, str):
    print(df_results)

elif df_results.empty:
    print("No conjunction results found.")

else:
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=list(df_results.columns),
            fill_color='#1f2937',
            font=dict(color='white', size=14),
            align='center',
            height=40
        ),
        cells=dict(
            values=df_results.T.values,  # <-- FIX: ensures correct structure
            fill_color=[
                ['#111827', '#1f2937'] * (len(df_results)//2 + 1)
            ],
            font=dict(color='white', size=12),
            align='center',
            height=32
        )
    )])

    fig.update_layout(
        title="Top 10 Nearest Space Objects (Next 7 Days)",
        template="plotly_dark",
        margin=dict(l=10, r=10, t=50, b=10)
    )

    fig.show()


Fetching live TLEs from Celestrak...
Target identified: STARLINK-1661
Orbit Regime: LEO
Scanning 14063 other LEO objects for the next 7 days...

Top 10 Nearest Space Objects (Next 7 Days):


In [3]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from skyfield.api import load

import os

# Substitua pelo caminho real onde o seu CUDA está instalado
cuda_path = r'C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.2\bin' 
if os.path.exists(cuda_path):
    os.add_dll_directory(cuda_path)


# Tenta importar CuPy para rodar na GPU, senão usa NumPy (CPU)
try:
    import cupy as cp
    print("🚀 GPU Detectada! Usando CuPy para aceleração.")
    USING_GPU = True
except ImportError:
    import numpy as cp
    print("💻 GPU não encontrada. Usando NumPy vetorizado (Otimizado).")
    USING_GPU = False

CELESTRAK_URL = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=active&FORMAT=tle'

def get_orbit_regime(sat):
    # Otimizado: Acesso direto ao modelo sem try/except repetitivo
    no = getattr(sat.model, 'no_kozai', getattr(sat.model, 'no', 0))
    revs_per_day = no * 1440 / (2 * np.pi)
    
    if revs_per_day > 11.25: return "LEO"
    if 1.05 < revs_per_day <= 11.25: return "MEO"
    if 0.9 <= revs_per_day <= 1.05: return "GEO"
    return "Outro"

def run_optimized_analysis(target_id, days=7, step_minutes=15):
    ts = load.timescale()
    now = datetime.utcnow()
    
    # 1. Carga de Dados
    satellites = load.tle_file(CELESTRAK_URL)
    by_number = {sat.model.satnum: sat for sat in satellites}
    
    if target_id not in by_number:
        return f"ID {target_id} não encontrado."
    
    target = by_number[target_id]
    regime = get_orbit_regime(target)
    
    # 2. Filtragem de Candidatos (Mesmo regime)
    candidates = [s for s in satellites if s.model.satnum != target_id and get_orbit_regime(s) == regime]
    num_candidates = len(candidates)
    
    # 3. Vetorização de Tempo
    time_range = range(0, int(days * 24 * 60), step_minutes)
    times = ts.utc(now.year, now.month, now.day, now.hour, time_range)
    num_steps = len(times)

    # 4. Propagação do Alvo (Uma única vez)
    # Posição do alvo formatada como (3, passos_tempo) -> (x, y, z)
    target_pos = cp.array(target.at(times).position.km) 

    # 5. Cálculo Vetorizado de Distâncias (Coração da Otimização)
    # Vamos processar em blocos para evitar estourar a memória da GPU se houver muitos satélites
    batch_size = 500 
    all_min_dists = []
    all_min_times = []

    print(f"Analisando {num_candidates} objetos em {num_steps} pontos no tempo...")

    for i in range(0, num_candidates, batch_size):
        batch = candidates[i : i + batch_size]
        
        # Coleta as posições de todos os satélites do bloco de uma vez
        # Criamos um tensor de formato (N_sats, 3, N_passos)
        batch_positions = np.array([s.at(times).position.km for s in batch])
        batch_positions_gpu = cp.array(batch_positions)

        # Cálculo da distância Euclidiana em 3D: sqrt(sum(diff^2))
        # Transmitimos (broadcast) a posição do alvo para subtrair de todos os candidatos simultaneamente
        # diff shape: (N_sats, 3, N_passos)
        diff = batch_positions_gpu - target_pos[np.newaxis, :, :]
        
        # Quadrado das diferenças e soma no eixo das coordenadas (eixo 1)
        dist_sq = cp.sum(cp.square(diff), axis=1)
        distances = cp.sqrt(dist_sq)

        # Encontra o índice da menor distância para cada satélite no bloco
        min_indices = cp.argmin(distances, axis=1)
        
        # Extrai os valores mínimos
        for idx_in_batch, time_idx in enumerate(min_indices):
            min_d = float(distances[idx_in_batch, time_idx])
            all_min_dists.append(min_d)
            all_min_times.append(times[int(time_idx)])

    # 6. Consolidação dos Resultados
    results = []
    for idx, sat in enumerate(candidates):
        results.append({
            'NAME': sat.name,
            'NORAD_ID': sat.model.satnum,
            'MIN_DIST_KM': round(all_min_dists[idx], 2),
            'TIME_UTC': all_min_times[idx].utc_strftime('%Y-%m-%d %H:%M:%S')
        })

    return pd.DataFrame(results).sort_values('MIN_DIST_KM').head(10)

# Teste com Starlink ou ISS
df_results = run_optimized_analysis(25544) # ISS
print("\nTop 10 Aproximações (Otimizado):")
print(df_results.to_string(index=False))

🚀 GPU Detectada! Usando CuPy para aceleração.
Analisando 14063 objetos em 672 pontos no tempo...


DynamicLibNotFoundError: Failure finding "nvrtc*.dll": No such file: nvrtc*.dll, No such file: nvrtc*.dll
